# Initial Processing
This is in a notebook for one main reason: cropping boundaries varied across all videos, so there was a lot of trial and error involved in finding them. 

Make sure you have all the required packages listed in the below cell, then run the below cell once.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

#helper function: for numbering images
def GetNumberString(x):
    assert type(x) == int
    x_str = str(x)
    assert len(x_str) <= 4 and x > 0
    if len(x_str) == 1:
        return "000" + x_str
    elif len(x_str) == 2:
        return "00" + x_str
    elif len(x_str) == 3:
        return "0" + x_str
    return x_str

#helper function: converting unruly filenames into a neatly formatted string
#with only underscore separations
def FormatFileName(fname):
    new_fname = ""
    for c in fname:
        if c == " " or c == ".":
            new_fname += "_"
        else:
            new_fname += c
    return new_fname[:-4] #remove file extension such as .png, .mp4, .mov (change if needed)

Change the below directories so that `data_dir` has the path to where your unprocessed video data is located, and `image_dir` has the path you want to place the processed images. This cell once again only needs to be run one time.

In [ ]:
data_dir = "/Users/jonathanzhu/nematostella_videos/vids/"
image_dir = "/Users/jonathanzhu/nematostella_videos/imgs/"
files = os.listdir(data_dir)
files #you can comment out this line, but this will show you all the videos in your video data folder

For every video you want to process in your video folder, you'll need to change the number in the first line of the below cell and run the cell every time you change the number.

In [ ]:
file = files[0] #change the number in the square brackets
print(file)

#the below print statements can be uncommented
cap = cv2.VideoCapture(data_dir + file)
#print(cap.get(cv2.CAP_PROP_FPS))

fps = cap.get(cv2.CAP_PROP_FPS)
#frames_in_min = 1800
frames_in_min = int(fps * 60)
#print(frames_in_min)

The below cell should also be run every time you want to process a new video. It will cut off the first six seconds, then get a grayscale version of the first frame.

In [ ]:
frames_in_six_secs = fps * 6
frame_num = 0
while frame_num < frames_in_six_secs:
    ret, frame = cap.read()
    frame_num += 1

#GRAYSCALE
plt.imshow(frame)
#cv2.imwrite(image_dir + "test.png", frame)
gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
plt.imshow(gray)
#cv2.imwrite(image_dir + "test_gray.png", gray)

cap.release()
cv2.destroyAllWindows()


Based on the above, you'll need to alter the numbers for `x_min`, `x_max`, `y_min`, and `y_max` to properly crop your image as desired. This will take some trial and error, but the below cell will show you how the frame saved from the above cell looks at your crop levels.

In [ ]:
x_min = 25
x_max = 1040
y_min = 695
y_max = 1705

#cropping: done with numpy array indices
crop = gray[x_min:x_max, y_min:y_max] 
plt.imshow(crop)
#cv2.imwrite(image_dir + "test_crop.png", crop)


Once you're satisfied with your crop levels, you can run the below cell to convert the video into a folder of images. 

In [ ]:
#combine into one
current_dir = image_dir + FormatFileName(file) + "/"
os.mkdir(current_dir)

cap = cv2.VideoCapture(data_dir + file)

#cut first 6 seconds
frames_in_six_secs = fps * 6
frame_num = 0
while frame_num < frames_in_six_secs:
    ret, frame = cap.read()
    frame_num += 1

file_num = 1
while (file_num < frames_in_min):
    ret, frame = cap.read()

    if not ret:
        break

    #GRAYSCALE
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)

    #cropping: done with numpy array indices
    crop = gray[x_min:x_max, y_min:y_max]

    #save final image
    cv2.imwrite(current_dir + FormatFileName(file) + GetNumberString(file_num) + ".png", crop)
    file_num += 1
    

print("Number of images: " + str(file_num))
cap.release()
cv2.destroyAllWindows()